# N00 · El arnés de experimentos

## Pregunta

> ¿Puedo lanzar un experimento nuevo cambiando **una línea de configuración**, en vez de
> copiar el notebook anterior?

## Hipótesis

*(Escríbela antes de ejecutar nada, y no la edites después.)*

Creo que sí, y que el coste de construirlo se amortiza a partir del quinto experimento.
Espero que lo difícil no sea el bucle de entrenamiento —eso es estándar— sino decidir
**qué guardar** para que un resultado sea reproducible dentro de seis meses.

---

## Por qué este notebook es el primero

No enseña nada de deep learning. Construye la infraestructura que hace que los otros
treinta sean rentables.

Sin arnés, cada experimento cuesta una tarde: copiar el notebook anterior, cambiar dos
números, perder la pista de cuál era cuál. Con arnés, cuesta cinco minutos. **Esa
diferencia decide si harás diez experimentos o cien.**

> ⚠️ **La tentación es saltárselo.** Es el notebook menos vistoso del itinerario y el
> que más rendimiento da.

## Las seis decisiones de diseño

Antes del código, lo que hay detrás. Cada decisión se puede discutir; lo que no se puede
es no tomarla.

### 1. El experimento es un diccionario

```python
{"dataset": "recta", "modelo": "mlp", "optim_args": {"lr": 1e-2}, "epocas": 30}
```

Todo lo que **cambia** entre experimentos vive en el config. Todo lo que **no cambia**
vive en el arnés. Si tengo que editar el arnés para lanzar una variante, el arnés está
mal diseñado.

### 2. Registro por nombre, no por importación

El config dice `"modelo": "mlp"`, no importa una clase. Añadir un modelo nuevo es
decorar una función:

```python
@H.modelo("cnn")
def cnn(...): ...
```

Así el config es **JSON serializable**, y eso es lo que permite guardarlo en disco y
compararlo después.

### 3. Ganchos (callbacks) desde el día uno

En N06 voy a añadir diagnósticos: histogramas de pesos, normas de gradiente, test de
sobreajuste. Si no dejo el hueco ahora, tendré que reescribir el bucle entonces.

La clase `Callback` está vacía a propósito. Es un contrato, no una implementación.

### 4. Guardar en disco siempre, no en memoria

Un notebook se cierra y pierdes todo. Cada ejecución escribe cuatro ficheros:

| Fichero | Para qué |
|---|---|
| `config.json` | Qué lancé |
| `metrics.csv` | Qué salió |
| `weights.pt` | Poder retomarlo (N07) |
| `meta.json` | Semilla, fecha, commit, versiones |

`meta.json` es el que casi nadie guarda y el que más falta hace a los seis meses.

### 5. La semilla es un argumento, no una constante

Porque en N13 voy a lanzar el mismo experimento con cinco semillas para medir mi ruido
de fondo. Si la semilla está fija dentro del código, ese notebook no se puede hacer.

### 6. Menos de 300 líneas, y ni una más

**Esto no es un framework.** En cuanto empiece a crecer, estaré construyendo
infraestructura en vez de aprendiendo. Si necesito más, es que me he desviado.

---

## El arnés

Se escribe en `lab/harness.py`, **no en el notebook**. Motivo: los otros treinta
notebooks lo van a importar. Si vive en una celda, acabo copiándolo, y las copias
divergen.

El notebook lo **construye** y lo **prueba**; el fichero es el artefacto.

### Primero: el arranque que llevarán los 31 notebooks

Cuatro líneas que suben hasta la raíz del proyecto. Con esto da igual desde dónde abras
el notebook: `lab/` siempre se importa y `runs/` siempre se crea en el mismo sitio.

Sin esto, el notebook 12 guardaría resultados en una carpeta distinta que el 3, y los
perderías sin darte cuenta.

In [ ]:
# ── ARRANQUE ── (copiar tal cual en todos los notebooks)
import os, sys
from pathlib import Path

while not (Path.cwd() / "lab").exists() and Path.cwd() != Path.cwd().parent:
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

print("raíz del proyecto:", Path.cwd())

In [ ]:
Path("lab").mkdir(exist_ok=True)
Path("lab/__init__.py").touch()
print("carpeta lab/ lista")

In [ ]:
%%writefile lab/harness.py
"""Experiment harness: turns a config dict into results on disk.

ES: Arnés de experimentos. Un experimento es un diccionario; esto lo convierte
en resultados en disco.

Design rule / Regla de diseño:
    Everything that CHANGES between experiments lives in the config.
    Everything that DOESN'T lives here.

Comment convention / Convenio de comentarios:
    Docstrings in English describe WHAT. Spanish notes explain WHY.
"""
from __future__ import annotations

import json
import platform
import random
import subprocess
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Callable, Iterable

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.optim import Optimizer
from torch.utils.data import DataLoader

RUNS_DIR = Path("runs")
DEFAULT_LOSS = "mse_loss"

ModelBuilder = Callable[..., nn.Module]
DatasetBuilder = Callable[..., tuple[DataLoader, DataLoader]]
OptimizerBuilder = Callable[..., Optimizer]


# ─────────────────────────────────────────────────────────────────────────────
# Registry
# ─────────────────────────────────────────────────────────────────────────────
class Registry(dict):
    """Maps a name to a builder, so configs stay JSON-serializable.

    ES: El config dice "mlp" en vez de importar una clase. Eso es lo que
    permite guardarlo en disco y compararlo después.
    """

    def __init__(self, kind: str) -> None:
        super().__init__()
        self.kind = kind

    def register(self, name: str) -> Callable:
        def decorator(builder):
            self[name] = builder
            return builder
        return decorator

    def build(self, name: str, **kwargs):
        if name not in self:
            raise KeyError(f"unknown {self.kind} '{name}'. Available: {sorted(self)}")
        return self[name](**kwargs)


models = Registry("model")
datasets = Registry("dataset")
optimizers = Registry("optimizer")


# ─────────────────────────────────────────────────────────────────────────────
# Reproducibility
# ─────────────────────────────────────────────────────────────────────────────
def set_seed(seed: int, deterministic: bool = True) -> None:
    """Seed every random source we know about.

    ES: "Todas las que conocemos" no es "todas". En GPU quedan operaciones no
    deterministas aunque la semilla sea idéntica. Eso se mide en N13, no se
    asume aquí.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def current_git_commit() -> str:
    """Short commit hash, or 'no-git' when unavailable."""
    try:
        result = subprocess.run(
            ["git", "rev-parse", "--short", "HEAD"],
            capture_output=True, text=True, timeout=2,
        )
        return result.stdout.strip() or "no-git"
    except Exception:
        return "no-git"


# ─────────────────────────────────────────────────────────────────────────────
# Data structures
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class Components:
    """Everything a training loop needs, built from a config."""

    train_loader: DataLoader
    val_loader: DataLoader
    model: nn.Module
    optimizer: Optimizer
    loss_fn: Callable


@dataclass
class TrainingState:
    """What callbacks can inspect while training runs.

    ES: Existe para que N06 pueda enchufar diagnósticos SIN tocar el bucle.
    """

    config: dict
    model: nn.Module
    optimizer: Optimizer
    device: torch.device
    epoch: int = 0
    step: int = 0
    history: list[dict] = field(default_factory=list)
    scratch: dict = field(default_factory=dict)


@dataclass
class ExperimentResult:
    """Outcome of a single run."""

    run_id: str
    config: dict
    seed: int
    history: list[dict]
    model: nn.Module
    elapsed_seconds: float
    scratch: dict = field(default_factory=dict)

    @property
    def final_metrics(self) -> dict:
        return self.history[-1]

    def metric(self, name: str = "val_loss") -> float:
        return self.final_metrics[name]


class Callback:
    """Empty hooks. N06 will subclass this for diagnostics.

    ES: Es un contrato, no una implementación. Está vacío a propósito.
    """

    def on_train_start(self, state: TrainingState) -> None: ...
    def on_batch_end(self, state: TrainingState, loss: float) -> None: ...
    def on_epoch_end(self, state: TrainingState) -> None: ...
    def on_train_end(self, state: TrainingState) -> None: ...


# ─────────────────────────────────────────────────────────────────────────────
# Training
# ─────────────────────────────────────────────────────────────────────────────
def build_components(config: dict, seed: int) -> Components:
    """Instantiate data, model and optimizer from the config."""
    set_seed(seed)
    train_loader, val_loader = datasets.build(
        config["dataset"], **config.get("dataset_args", {}))
    model = models.build(config["model"], **config.get("model_args", {}))
    optimizer = optimizers.build(
        config.get("optimizer", "adam"),
        params=model.parameters(),
        **config.get("optimizer_args", {"lr": 1e-3}),
    )
    loss_fn = getattr(torch.nn.functional, config.get("loss", DEFAULT_LOSS))
    return Components(train_loader, val_loader, model, optimizer, loss_fn)


def resolve_device(config: dict) -> torch.device:
    requested = config.get("device") or ("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(requested)


def train_one_epoch(components: Components, state: TrainingState,
                    callbacks: Iterable[Callback]) -> float:
    """Run one full pass over the training set. Returns mean loss."""
    components.model.train()
    total_loss, seen = 0.0, 0

    for inputs, targets in components.train_loader:
        inputs = inputs.to(state.device)
        targets = targets.to(state.device)

        components.optimizer.zero_grad()
        loss = components.loss_fn(components.model(inputs), targets)
        loss.backward()
        components.optimizer.step()

        state.step += 1
        total_loss += loss.item() * len(inputs)
        seen += len(inputs)
        for callback in callbacks:
            callback.on_batch_end(state, loss.item())

    return total_loss / seen


@torch.no_grad()
def evaluate(components: Components, device: torch.device) -> dict:
    """Mean loss over the validation set."""
    components.model.eval()
    total_loss, seen = 0.0, 0
    for inputs, targets in components.val_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)
        total_loss += components.loss_fn(components.model(inputs), targets).item() * len(inputs)
        seen += len(inputs)
    return {"loss": total_loss / max(seen, 1)}


def run_experiment(config: dict, seed: int | None = None,
                   callbacks: list[Callback] | None = None,
                   save: bool = True, verbose: bool = True) -> ExperimentResult:
    """Train one model end to end and return its result.

    ES: Lanzar dos veces el mismo config con la misma semilla debe dar
    exactamente lo mismo. Si no, la semilla no llega a algún sitio.
    """
    started_at = time.time()
    seed = config.get("seed", 0) if seed is None else seed
    callbacks = callbacks or []
    device = resolve_device(config)

    components = build_components(config, seed)
    components.model.to(device)

    state = TrainingState(config=config, model=components.model,
                          optimizer=components.optimizer, device=device)
    for callback in callbacks:
        callback.on_train_start(state)

    total_epochs = config["epochs"]
    log_every = max(1, total_epochs // 5)

    for epoch in range(total_epochs):
        state.epoch = epoch
        train_loss = train_one_epoch(components, state, callbacks)
        val_metrics = evaluate(components, device)

        state.history.append({"epoch": epoch, "train_loss": train_loss,
                              **{f"val_{k}": v for k, v in val_metrics.items()}})
        for callback in callbacks:
            callback.on_epoch_end(state)

        if verbose and (epoch % log_every == 0 or epoch == total_epochs - 1):
            print(f"  epoch {epoch:3d}  train {train_loss:.5f}  val {val_metrics['loss']:.5f}")

    for callback in callbacks:
        callback.on_train_end(state)

    result = ExperimentResult(
        run_id=make_run_id(config, seed),
        config=config, seed=seed, history=state.history,
        model=components.model, scratch=state.scratch,
        elapsed_seconds=round(time.time() - started_at, 2),
    )
    if save:
        save_run(result)
    return result


def make_run_id(config: dict, seed: int) -> str:
    name = config.get("name", "run")
    return f"{name}_s{seed}_{time.strftime('%Y%m%d-%H%M%S')}"


# ─────────────────────────────────────────────────────────────────────────────
# Persistence
# ─────────────────────────────────────────────────────────────────────────────
def save_run(result: ExperimentResult) -> Path:
    """Write config, metrics, weights and metadata.

    ES: meta.json es el que casi nadie guarda y el que más falta hace a los
    seis meses.
    """
    run_path = RUNS_DIR / result.run_id
    run_path.mkdir(parents=True, exist_ok=True)

    (run_path / "config.json").write_text(json.dumps(result.config, indent=2))
    pd.DataFrame(result.history).to_csv(run_path / "metrics.csv", index=False)
    torch.save(result.model.state_dict(), run_path / "weights.pt")
    (run_path / "meta.json").write_text(json.dumps({
        "run_id": result.run_id,
        "seed": result.seed,
        "date": time.strftime("%Y-%m-%d %H:%M:%S"),
        "commit": current_git_commit(),
        "elapsed_seconds": result.elapsed_seconds,
        "torch": torch.__version__,
        "python": platform.python_version(),
        "host": platform.node(),
    }, indent=2))
    return run_path


def load_run(run_id: str) -> dict:
    """Read a saved run back from disk."""
    run_path = RUNS_DIR / run_id
    return {
        "run_id": run_id,
        "config": json.loads((run_path / "config.json").read_text()),
        "meta": json.loads((run_path / "meta.json").read_text()),
        "history": pd.read_csv(run_path / "metrics.csv"),
    }


def list_runs(pattern: str = "") -> list[str]:
    if not RUNS_DIR.exists():
        return []
    return sorted(p.name for p in RUNS_DIR.iterdir() if p.is_dir() and pattern in p.name)


# ─────────────────────────────────────────────────────────────────────────────
# Comparing
# ─────────────────────────────────────────────────────────────────────────────
def compare_runs(run_ids: list[str], metric: str = "val_loss") -> pd.DataFrame:
    """One row per run, with its config and final metric."""
    rows = []
    for run_id in run_ids:
        run = load_run(run_id)
        flat_config = {k: v for k, v in run["config"].items() if not isinstance(v, dict)}
        rows.append({
            "run_id": run_id,
            "seed": run["meta"]["seed"],
            **flat_config,
            "final": run["history"][metric].iloc[-1],
            "best": run["history"][metric].min(),
            "seconds": run["meta"]["elapsed_seconds"],
        })
    return pd.DataFrame(rows)


def plot_runs(run_ids: list[str], metrics=("train_loss", "val_loss"),
              log_scale: bool = True, ax=None):
    """Overlay learning curves from several runs."""
    import matplotlib.pyplot as plt

    if ax is None:
        _, ax = plt.subplots(figsize=(7, 4))

    for run_id in run_ids:
        history = load_run(run_id)["history"]
        label = run_id.split("_")[0]
        for index, metric in enumerate(metrics):
            if metric in history:
                ax.plot(history["epoch"], history[metric],
                        label=f"{label} · {metric}",
                        linestyle="-" if index == 0 else "--", alpha=0.9)

    if log_scale:
        ax.set_yscale("log")
    ax.set_xlabel("epoch")
    ax.set_ylabel("loss")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    return ax


# ─────────────────────────────────────────────────────────────────────────────
# Sweeps
# ─────────────────────────────────────────────────────────────────────────────
def _with_override(config: dict, key: str, value) -> dict:
    """Copy of config with one key replaced. Supports 'optimizer_args.lr'."""
    updated = json.loads(json.dumps(config))
    if "." in key:
        section, inner_key = key.split(".", 1)
        updated.setdefault(section, {})[inner_key] = value
    else:
        updated[key] = value
    return updated


def sweep(base_config: dict, key: str, values: list, seeds=(0,)) -> list[str]:
    """Run the same config varying ONE key. Returns the run ids."""
    run_ids = []
    for value in values:
        for seed in seeds:
            config = _with_override(base_config, key, value)
            config["name"] = f"{base_config.get('name', 'run')}-{key.split('.')[-1]}{value}"
            print(f"▶ {key}={value}  seed={seed}")
            run_ids.append(run_experiment(config, seed=seed, verbose=False).run_id)
    return run_ids


def repeat_with_seeds(config: dict, n_seeds: int = 5,
                      metric: str = "val_loss") -> pd.DataFrame:
    """Same experiment, several seeds. Prints the spread.

    ES: Esa dispersión es tu umbral de credibilidad: por debajo de ella,
    ninguna diferencia es un resultado. Se mide en serio en N13.
    """
    run_ids = []
    for seed in range(n_seeds):
        config = dict(config, name=f"{config.get('name', 'run')}-rep")
        print(f"▶ seed {seed}")
        run_ids.append(run_experiment(config, seed=seed, verbose=False).run_id)

    table = compare_runs(run_ids, metric)
    mean, std = table["final"].mean(), table["final"].std()
    print(f"\n{metric}: mean {mean:.5f} · std {std:.5f} · "
          f"range [{table['final'].min():.5f}, {table['final'].max():.5f}]")
    print(f"→ A difference smaller than ~{2 * std:.5f} is NOT a result.")
    return table

---

## Un problema de juguete para probarlo

**No es el dataset del itinerario.** Ese llega en N01. Aquí solo necesito algo que
entrene en dos segundos para comprobar que la tubería funciona.

Recta con ruido: $y = 3x + 2 + \varepsilon$. La elijo porque **conozco la respuesta
verdadera**, así que puedo saber si el modelo está cerca o lejos, no solo si mejora.

In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from lab import harness as H

GROUND_TRUTH = {"slope": 3.0, "intercept": 2.0}   # lo que el modelo debe descubrir
NOISE_STD = 0.5


@H.datasets.register("line")
def build_line_dataset(n_samples=512, slope=3.0, intercept=2.0,
                       noise_std=NOISE_STD, seed=0, batch_size=32):
    rng = np.random.default_rng(seed)
    x = rng.uniform(-3, 3, size=(n_samples, 1)).astype("float32")
    y = (slope * x[:, 0] + intercept + rng.normal(0, noise_std, n_samples)).astype("float32")
    split = int(0.8 * n_samples)

    def make_loader(start, end, shuffle):
        subset = TensorDataset(torch.tensor(x[start:end]), torch.tensor(y[start:end]))
        return DataLoader(subset, batch_size=batch_size, shuffle=shuffle)

    return make_loader(0, split, True), make_loader(split, n_samples, False)


@H.models.register("mlp")
def build_mlp(input_size=1, hidden_size=16, output_size=1, n_hidden_layers=1):
    layers, size = [], input_size
    for _ in range(n_hidden_layers):
        layers += [nn.Linear(size, hidden_size), nn.ReLU()]
        size = hidden_size
    layers.append(nn.Linear(size, output_size))
    return nn.Sequential(*layers, nn.Flatten(0))


@H.optimizers.register("adam")
def build_adam(params, **kwargs):
    return torch.optim.Adam(params, **kwargs)


@H.optimizers.register("sgd")
def build_sgd(params, **kwargs):
    return torch.optim.SGD(params, **kwargs)


print("registered →", list(H.datasets), list(H.models), list(H.optimizers))

### El techo: qué error es el mejor posible

Con ruido $\sigma = 0.5$, ningún modelo puede bajar de $\sigma^2 = 0.25$ de error
cuadrático medio. Ese es el **error irreducible**.

Saberlo cambia cómo se lee un resultado: `0.24` no es "bueno", es **prácticamente
óptimo**. Sin este número no sabría distinguir un 0.24 excelente de un 0.24 mediocre.

> Esta idea —que cada dataset traiga su techo— es lo que sistematizaré en N01.

In [ ]:
irreducible_error = NOISE_STD ** 2
print(f"irreducible error (best possible MSE) = {irreducible_error:.4f}")

---

## Prueba 1 · Una ejecución

Si esto funciona, la tubería entera funciona.

In [ ]:
base_config = {
    "name": "n00",
    "dataset": "line",
    "dataset_args": {"n_samples": 512, "noise_std": NOISE_STD},
    "model": "mlp",
    "model_args": {"hidden_size": 16, "n_hidden_layers": 1},
    "optimizer": "adam",
    "optimizer_args": {"lr": 1e-2},
    "epochs": 30,
    "loss": "mse_loss",
}

result = H.run_experiment(base_config, seed=0)
print(f"\nfinal: {result.metric():.4f}   (floor: {irreducible_error:.4f})")
print(f"saved to: runs/{result.run_id}/")

In [ ]:
# ¿Encontró la recta verdadera? / Did it find the true line?
model = result.model.eval()
with torch.no_grad():
    estimated_intercept = model(torch.tensor([[0.0]])).item()
    estimated_slope = model(torch.tensor([[1.0]])).item() - estimated_intercept

print(f"slope     estimated {estimated_slope:.3f}  (true {GROUND_TRUTH['slope']})")
print(f"intercept estimated {estimated_intercept:.3f}  (true {GROUND_TRUTH['intercept']})")

### Un detalle que no esperaba

La pérdida de validación puede quedar **por debajo** del techo teórico. No es un error:
el techo $\sigma^2$ es una **esperanza**, y el conjunto de validación tiene ~100
muestras. El ruido empírico de esa muestra concreta fluctúa alrededor de 0.25.

Dos consecuencias que valen para todo el itinerario:

- **Un techo teórico es una referencia, no una barrera dura.** Bajar de él en un
  conjunto pequeño es azar, no genialidad.
- **Si veo un resultado que bate el techo por mucho, es fuga de datos**, no un
  descubrimiento (→ N15).

Anótalo en la bitácora, en «qué me sorprendió».

---

## Prueba 2 · ¿Qué se guardó?

Un resultado que no se puede reconstruir dentro de seis meses no es un resultado.

In [ ]:
import json

run_path = Path("runs") / result.run_id
print("files:", sorted(p.name for p in run_path.iterdir()), "\n")
print(json.dumps(json.loads((run_path / "meta.json").read_text()), indent=2))

---

## Prueba 3 · Comparar sin escribir código

La prueba de fuego del arnés: lanzar tres variantes **cambiando solo un valor**.

In [ ]:
run_ids = H.sweep(base_config, "optimizer_args.lr", [1e-1, 1e-2, 1e-3], seeds=(0,))
H.compare_runs(run_ids)

In [ ]:
import matplotlib.pyplot as plt

H.plot_runs(run_ids, metrics=("val_loss",))
plt.axhline(irreducible_error, color="k", ls=":", lw=1, label="irreducible error")
plt.legend(fontsize=8)
plt.title("Learning rate · validation loss")
plt.show()

**Lectura.** La línea punteada es el techo. Una tasa demasiado baja no llega ni cerca en
30 épocas; una demasiado alta oscila. Esa gráfica es N09 entero, adelantado — y ha
costado dos líneas porque el arnés ya estaba.

---

## 🔨 Qué rompo aquí · El test de reproducibilidad

El arnés no sirve de nada si no puedo confiar en que **la misma semilla da el mismo
resultado**. Dos comprobaciones, y las dos tienen que pasar:

1. Misma semilla → resultado **idéntico**
2. Semilla distinta → resultado **distinto**

La segunda parece trivial y no lo es: si fallara, significaría que la semilla no está
llegando a algún sitio y estaría comparando ejecuciones que en realidad son la misma.

In [ ]:
def final_loss(seed):
    return H.run_experiment(base_config, seed=seed, save=False, verbose=False).metric()

first_run_seed_0 = final_loss(0)
second_run_seed_0 = final_loss(0)
run_seed_1 = final_loss(1)

print(f"seed 0, run 1 : {first_run_seed_0:.12f}")
print(f"seed 0, run 2 : {second_run_seed_0:.12f}")
print(f"seed 1        : {run_seed_1:.12f}\n")

assert first_run_seed_0 == second_run_seed_0, \
    "FAIL: same seed gave different results. The seed is not reaching somewhere."
assert first_run_seed_0 != run_seed_1, \
    "FAIL: different seeds gave the same result. The seed is not being used."

print("✓ reproducible with the same seed")
print("✓ sensitive to the seed")

### El matiz que hay que dejar anotado

Esto pasa **en CPU**. En GPU algunas operaciones son no deterministas aunque la semilla
sea idéntica, y `torch.backends.cudnn.deterministic = True` no siempre basta.

> **"Semilla fija" no equivale a "reproducible".** Cuando tenga GPU, este mismo test hay
> que repetirlo. Es una de las cosas que se miden en serio en **N13**.

---

## Prueba 4 · La herramienta que usaré en N13

El mismo experimento, cinco semillas. No para sacar conclusiones —eso es N13— sino para
comprobar que la herramienta existe y funciona.

In [ ]:
seed_table = H.repeat_with_seeds(base_config, n_seeds=5)
seed_table[["run_id", "seed", "final"]]

**Guarda ese número.** La dispersión entre semillas es tu **umbral de credibilidad**:
cualquier diferencia menor que eso, en cualquier experimento futuro, no es un resultado.

Aquí sale de un problema de juguete. En N13 se mide de verdad.

---

## Criterio de terminado

- [x] Lanzo tres configuraciones cambiando solo un diccionario
- [x] Comparo las tres en una tabla y en una gráfica sin escribir código nuevo
- [x] Cada ejecución deja config, métricas, pesos y metadatos en disco
- [x] Misma semilla → resultado idéntico; semilla distinta → resultado distinto
- [x] Hay un hueco (`Callback`) para los diagnósticos de N06
- [x] El arnés cabe en menos de 300 líneas

---

## Cierre de bitácora

*(Copiar a la entrada de bitácora antes de cerrar el notebook.)*

### Qué aprendí

### Qué me sorprendió

> El campo más valioso. Si está vacío, o el experimento era demasiado seguro, o no me
> he fijado bastante.

### Qué haría distinto

### Siguiente paso

**N01 · Fábrica de datos sintéticos.** Con dos cosas heredadas de aquí:

1. Cada generador debe traer su **techo teórico**, como el $\sigma^2$ de este notebook.
2. Los generadores se registran con `@H.dataset`, así que N01 solo añade funciones: no
   toca el arnés.